# convert_tract_wacs_to_puma.ipynb

This notebook runs after get_wacs_data runs. It takes the tract-level data in the output of that notebook and aggregates them to get puma-level statistics. 

In [ ]:
import pandas as pd

In [2]:
df = pd.read_csv("wac_tracts_2017.csv").set_index("w_geocode")
df

,C000,CA01,CA02,CA03,CE01,CE02,CE03,CNS01,CNS02,CNS03,...,CNS15,CNS16,CNS17,CNS18,CNS19,CNS20,CD01,CD02,CD03,CD04
w_geocode,,,,,,,,,,,,,,,,,,,,,
10010201001000,3,1,1,1,1,2,0,0,0,0,...,0,0,0,0,0,0,0,2,0,0
10010201001007,1,0,0,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
10010201001016,8,0,5,3,2,3,3,0,0,0,...,0,0,0,8,0,0,0,3,1,4
10010201001018,29,8,12,9,4,13,12,0,0,0,...,0,0,0,0,0,0,7,7,6,1
10010201001022,155,13,98,44,44,30,81,0,0,0,...,155,0,0,0,0,0,5,40,41,56
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
560459513003079,16,3,10,3,3,11,2,0,0,0,...,0,0,0,0,0,0,1,6,3,3
560459513003082,11,0,2,9,1,3,7,0,5,0,...,0,0,0,0,0,0,4,5,2,0
560459513003087,21,1,13,7,7,5,9,0,15,0,...,0,0,0,0,0,2,1,13,5,1


In [3]:
df.index = df.index.astype("str")
df["tract"] = df.index.str[:-4].str.zfill(11)
df["tract"]

w_geocode
10010201001000     01001020100
10010201001007     01001020100
10010201001016     01001020100
10010201001018     01001020100
10010201001022     01001020100
                      ...     
560459513003079    56045951300
560459513003082    56045951300
560459513003087    56045951300
560459513003104    56045951300
560459513003106    56045951300
Name: tract, Length: 2197491, dtype: object

In [ ]:
tract_to_puma = pd.read_csv("../geometry/equivalencies/tract_to_puma.txt", delimiter = ",", dtype=str)
tract_to_puma.head() 

,STATEFP,COUNTYFP,TRACTCE,PUMA5CE
0,01,001,020100,02100
1,01,001,020200,02100
2,01,001,020300,02100
3,01,001,020400,02100
4,01,001,020500,02100


In [5]:
tract_to_puma["tract"] = tract_to_puma["STATEFP"] + tract_to_puma["COUNTYFP"] + tract_to_puma["TRACTCE"]
tract_to_puma = tract_to_puma.set_index("tract")
tract_to_puma.head()

,STATEFP,COUNTYFP,TRACTCE,PUMA5CE
tract,,,,
01001020100,01,001,020100,02100
01001020200,01,001,020200,02100
01001020300,01,001,020300,02100
01001020400,01,001,020400,02100
01001020500,01,001,020500,02100


In [6]:
tract_to_puma["PUMA5CE"] = tract_to_puma["STATEFP"] + tract_to_puma["PUMA5CE"]
tract_to_puma["PUMA5CE"] = tract_to_puma["PUMA5CE"].str.zfill(7)
tract_to_puma.head()

,STATEFP,COUNTYFP,TRACTCE,PUMA5CE
tract,,,,
01001020100,01,001,020100,0102100
01001020200,01,001,020200,0102100
01001020300,01,001,020300,0102100
01001020400,01,001,020400,0102100
01001020500,01,001,020500,0102100


In [7]:
df["puma"] = tract_to_puma.loc[list(df["tract"])]["PUMA5CE"].values
df

,C000,CA01,CA02,CA03,CE01,CE02,CE03,CNS01,CNS02,CNS03,...,CNS17,CNS18,CNS19,CNS20,CD01,CD02,CD03,CD04,tract,puma
w_geocode,,,,,,,,,,,,,,,,,,,,,
10010201001000,3,1,1,1,1,2,0,0,0,0,...,0,0,0,0,0,2,0,0,01001020100,0102100
10010201001007,1,0,0,1,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,01001020100,0102100
10010201001016,8,0,5,3,2,3,3,0,0,0,...,0,8,0,0,0,3,1,4,01001020100,0102100
10010201001018,29,8,12,9,4,13,12,0,0,0,...,0,0,0,0,7,7,6,1,01001020100,0102100
10010201001022,155,13,98,44,44,30,81,0,0,0,...,0,0,0,0,5,40,41,56,01001020100,0102100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
560459513003079,16,3,10,3,3,11,2,0,0,0,...,0,0,0,0,1,6,3,3,56045951300,5600200
560459513003082,11,0,2,9,1,3,7,0,5,0,...,0,0,0,0,4,5,2,0,56045951300,5600200
560459513003087,21,1,13,7,7,5,9,0,15,0,...,0,0,0,2,1,13,5,1,56045951300,5600200


In [8]:
df = df.groupby(["puma"]).sum()
df.columns = ["TOT_JOBS", "JOBS_AGE_29", "JOBS_AGE_30_54", "JOBS_AGE_55", "JOBS_EARN_1250", "JOBS_EARN_1251_3333", "JOBS_EARN_3334", "AGR", "EXT", "UTL", "CON", "MFG", "WHL", "RET", "TRN", "INF", "FIN", "REL", "PRF", "MNG", "ADM", "EDU", "MED", "ENT", "FOD", "SRV", "PUB", "JOBS_EDU_NOHS", "JOBS_EDU_HS", "JOBS_EDU_NOBACH", "JOBS_EDU_BACH"]
df 

,TOT_JOBS,JOBS_AGE_29,JOBS_AGE_30_54,JOBS_AGE_55,JOBS_EARN_1250,JOBS_EARN_1251_3333,JOBS_EARN_3334,AGR,EXT,UTL,...,EDU,MED,ENT,FOD,SRV,PUB,JOBS_EDU_NOHS,JOBS_EDU_HS,JOBS_EDU_NOBACH,JOBS_EDU_BACH
puma,,,,,,,,,,,,,,,,,,,,,
0100100,65457,17347,34154,13956,16751,27438,21268,327,205,671,...,5368,8642,512,6493,1224,3114,6367,15886,15785,10072
0100200,53503,11576,29503,12424,10810,16538,26155,513,25,317,...,6841,3310,912,3905,883,3304,4480,11870,12913,12664
0100301,78052,19057,42152,16843,17911,20914,39227,144,62,68,...,4787,4209,644,8403,1261,769,5996,15325,18086,19588
0100302,71799,17583,38760,15456,19020,26195,26584,19,1,603,...,3229,18338,1169,6276,1744,3973,6161,15190,18302,14563
0100400,36988,8958,20085,7945,8844,17100,11044,335,21,456,...,3299,3988,119,2856,477,1770,3995,10033,9017,4985
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5600100,54571,11814,28972,13785,15032,18033,21506,911,1709,461,...,5514,7809,1726,9507,1706,3056,4773,13006,14332,10646
5600200,45007,8899,24461,11647,9757,13616,21634,659,6393,903,...,5045,5363,365,3784,1318,3866,3727,12044,13025,7312
5600300,62958,15643,32657,14658,15259,20965,26734,463,422,226,...,8652,9578,643,6302,1713,7923,4958,13439,16153,12765


In [9]:
df.to_csv("wac_puma_2017.csv")